# 🚀 Proyecto Final: Sistema de Gestión de Paquetería Inteligente


## 🧩 Descripción General
Se debe implementar un sistema de gestión de envíos para una empresa de mensajería y logística que permita registrar paquetes, asociarlos a clientes, llevar el control de su estado, y calcular costos según el tipo de envío. El sistema debe incluir relaciones entre clases (composición, agregación, asociación), uso de métodos mágicos, herencia, encapsulación, y polimorfismo.

### 🎯 Objetivo del Proyecto
- Evaluar los siguientes conceptos:

- Clases y objetos

- Atributos públicos, protegidos y privados

- Métodos de instancia, clase y estáticos

- Métodos mágicos (__str__, __init__, etc.)

- Encapsulación con validaciones (getters/setters)

- Asociación, agregación y composición

- Herencia y polimorfismo

- Buenas prácticas (modularidad, documentación)

### ESTRUCTURA E LAS CLASES

| Clase           | Rol                                                                 |
|----------------|----------------------------------------------------------------------|
| Cliente         | Representa a un usuario que envía o recibe paquetes                  |
| Paquete         | Clase base para paquetes, incluye atributos comunes y métodos mágicos |
| PaqueteExpress  | Hereda de Paquete, incluye recargo por urgencia                      |
| PaqueteEstandar | Hereda de Paquete, sin recargo                                       |
| Direccion       | Composición con Cliente (cada cliente tiene una dirección)           |
| Envio           | Agrega un paquete y se asocia a un cliente                           |
| SistemaEnvios   | Administra clientes y envíos                                         |


### 🧪 Requisitos funcionales
1. Registrar clientes y asociarles una dirección.

2. Crear paquetes (express o estándar).

3. Asignar paquetes a envíos y clientes.

4. Calcular el precio de cada envío.

5. Mostrar listado de envíos con información detallada (polimorfismo).

6. Validar datos (precio, peso, nombre del cliente, etc.).

7. Mostrar reporte de todos los envíos realizados.

8. Agregar seguimiento del paquete (pendiente, en tránsito, entregado) usando métodos set_estado().

9. Guardar la información en un archivo .txt o .json.

10. Mostrar totales de ventas por tipo de envío (estándar vs express).

11. Diseñar un menú de opciones para registrar clientes/envíos de forma interactiva.



# Solucion

In [ ]:
import json
from datetime import datetime

class Direccion:
    """Clase para manejar direcciones (composición con Cliente)"""
    def __init__(self, calle: str, ciudad: str, codigo_postal: str, pais: str):
        self.calle = calle
        self.ciudad = ciudad
        self.codigo_postal = codigo_postal
        self.pais = pais
    
    def __str__(self):
        return f"{self.calle}, {self.ciudad}, {self.codigo_postal}, {self.pais}"

class Cliente:
    """Clase para representar clientes del sistema"""
    def __init__(self, nombre: str, email: str, direccion: Direccion):
        self._nombre = nombre  # Atributo protegido
        self.email = email
        self.direccion = direccion  # Composición
        self._envios = []  # Asociación con Envio (lista protegida)
    
    @property
    def nombre(self):
        return self._nombre
    
    @nombre.setter
    def nombre(self, valor):
        if not valor.strip():
            raise ValueError("El nombre no puede estar vacío")
        self._nombre = valor
    
    def agregar_envio(self, envio):
        self._envios.append(envio)
    
    def __str__(self):
        return f"Cliente: {self._nombre} | Email: {self.email}"

class Paquete:
    """Clase base abstracta para paquetes"""
    def __init__(self, peso: float, dimensiones: tuple):
        self._peso = peso
        self._dimensiones = dimensiones
        self._estado = "pendiente"
        self._fecha_registro = datetime.now()
    
    @property
    def peso(self):
        return self._peso
    
    @peso.setter
    def peso(self, valor):
        if valor <= 0:
            raise ValueError("El peso debe ser positivo")
        self._peso = valor
    
    def set_estado(self, estado: str):
        estados_validos = ["pendiente", "en tránsito", "entregado"]
        if estado.lower() not in estados_validos:
            raise ValueError(f"Estado inválido. Use: {', '.join(estados_validos)}")
        self._estado = estado
    
    def calcular_costo(self):
        raise NotImplementedError("Método abstracto")
    
    def __str__(self):
        return f"{self.__class__.__name__}: {self._peso}kg, Estado: {self._estado}"

class PaqueteEstandar(Paquete):
    """Paquete estándar sin recargos"""
    def calcular_costo(self):
        return self._peso * 1000 + sum(self._dimensiones) * 500

class PaqueteExpress(Paquete):
    """Paquete express con recargo por urgencia"""
    def __init__(self, peso: float, dimensiones: tuple, urgencia: int = 1):
        super().__init__(peso, dimensiones)
        self._urgencia = urgencia
    
    @property
    def urgencia(self):
        return self._urgencia
    
    @urgencia.setter
    def urgencia(self, valor):
        if valor not in (1, 2, 3):
            raise ValueError("Urgencia debe ser 1, 2 o 3")
        self._urgencia = valor
    
    def calcular_costo(self):
        base = super().calcular_costo()
        return base * (1 + self._urgencia * 0.2)

class Envio:
    """Clase que gestiona envíos (agregación de Paquete)"""
    def __init__(self, cliente: Cliente, paquete: Paquete, destino: Direccion):
        self.cliente = cliente
        self.paquete = paquete
        self.destino = destino
        self._fecha_envio = datetime.now()
        cliente.agregar_envio(self)
    
    def __str__(self):
        return (f"Envío para {self.cliente.nombre}\n"
                f"Paquete: {self.paquete}\n"
                f"Costo: ${self.paquete.calcular_costo():,.2f}\n"
                f"Destino: {self.destino}")

class SistemaEnvios:
    """Clase principal del sistema"""
    _instance = None  # Para implementar Singleton
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.clientes = []
            cls._instance.envios = []
        return cls._instance
    
    def registrar_cliente(self, nombre: str, email: str, direccion: Direccion):
        if "@" not in email:
            raise ValueError("Email inválido")
        cliente = Cliente(nombre, email, direccion)
        self.clientes.append(cliente)
        return cliente
    
    def crear_envio(self, cliente: Cliente, tipo_paquete: str, peso: float, 
                   dimensiones: tuple, destino: Direccion, **kwargs):
        if tipo_paquete.lower() == "express":
            paquete = PaqueteExpress(peso, dimensiones, kwargs.get('urgencia', 1))
        else:
            paquete = PaqueteEstandar(peso, dimensiones)
        
        envio = Envio(cliente, paquete, destino)
        self.envios.append(envio)
        return envio
    
    def mostrar_envios(self):
        for i, envio in enumerate(self.envios, 1):
            print(f"\nEnvío #{i}")
            print("-" * 30)
            print(envio)
    
    def generar_reporte(self):
        total_estandar = sum(e.paquete.calcular_costo() 
                           for e in self.envios 
                           if isinstance(e.paquete, PaqueteEstandar))
        total_express = sum(e.paquete.calcular_costo() 
                          for e in self.envios 
                          if isinstance(e.paquete, PaqueteExpress))
        
        print("\nREPORTE DE VENTAS")
        print(f"Enviós estándar: ${total_estandar:,.2f}")
        print(f"Envíos express: ${total_express:,.2f}")
        print(f"TOTAL: ${total_estandar + total_express:,.2f}")
    
    def guardar_datos(self, archivo="envios.json"):
        datos = {
            "clientes": [{
                "nombre": c.nombre,
                "email": c.email,
                "direccion": vars(c.direccion)
            } for c in self.clientes],
            "envios": [{
                "cliente": e.cliente.nombre,
                "paquete": {
                    "tipo": e.paquete.__class__.__name__,
                    "peso": e.paquete.peso,
                    "dimensiones": e.paquete._dimensiones,
                    "estado": e.paquete._estado,
                    **({"urgencia": e.paquete.urgencia} 
                       if isinstance(e.paquete, PaqueteExpress) else {})
                },
                "destino": vars(e.destino),
                "fecha": e._fecha_envio.strftime("%Y-%m-%d %H:%M:%S")
            } for e in self.envios]
        }
        
        with open(archivo, "w") as f:
            json.dump(datos, f, indent=2)
    
    def cargar_datos(self, archivo="envios.json"):
        try:
            with open(archivo) as f:
                datos = json.load(f)
                
                # Reconstruir clientes
                clientes_map = {}
                for c_data in datos["clientes"]:
                    dir_data = c_data["direccion"]
                    direccion = Direccion(**dir_data)
                    cliente = self.registrar_cliente(
                        c_data["nombre"], 
                        c_data["email"], 
                        direccion
                    )
                    clientes_map[c_data["nombre"]] = cliente
                
                # Reconstruir envíos
                for e_data in datos["envios"]:
                    cliente = clientes_map[e_data["cliente"]]
                    p_data = e_data["paquete"]
                    
                    if p_data["tipo"] == "PaqueteExpress":
                        paquete = PaqueteExpress(
                            p_data["peso"],
                            tuple(p_data["dimensiones"]),
                            p_data["urgencia"]
                        )
                    else:
                        paquete = PaqueteEstandar(
                            p_data["peso"],
                            tuple(p_data["dimensiones"])
                        )
                    
                    paquete.set_estado(p_data["estado"])
                    destino = Direccion(**e_data["destino"])
                    
                    self.crear_envio(
                        cliente,
                        p_data["tipo"].lower(),
                        p_data["peso"],
                        tuple(p_data["dimensiones"]),
                        destino
                    )
                    
        except FileNotFoundError:
            print("No se encontró archivo de datos. Se creará uno nuevo.")

def menu_principal():
    sistema = SistemaEnvios()
    sistema.cargar_datos()
    
    while True:
        print("\n📦 SISTEMA DE PAQUETERÍA 📦")
        print("1. Registrar cliente")
        print("2. Crear envío")
        print("3. Mostrar envíos")
        print("4. Generar reporte")
        print("5. Cambiar estado paquete")
        print("6. Guardar datos")
        print("7. Salir")
        
        opcion = input("Seleccione una opción: ")
        
        if opcion == "1":
            print("\nREGISTRO DE CLIENTE")
            nombre = input("Nombre: ")
            email = input("Email: ")
            print("\nDirección:")
            calle = input("Calle: ")
            ciudad = input("Ciudad: ")
            cp = input("Código postal: ")
            pais = input("País: ")
            
            try:
                direccion = Direccion(calle, ciudad, cp, pais)
                sistema.registrar_cliente(nombre, email, direccion)
                print("✅ Cliente registrado")
            except Exception as e:
                print(f"❌ Error: {e}")
        
        elif opcion == "2":
            if not sistema.clientes:
                print("❌ No hay clientes registrados")
                continue
                
            print("\nNUEVO ENVÍO")
            print("Clientes disponibles:")
            for i, c in enumerate(sistema.clientes, 1):
                print(f"{i}. {c.nombre}")
            
            try:
                cliente_idx = int(input("Seleccione cliente: ")) - 1
                cliente = sistema.clientes[cliente_idx]
                
                tipo = input("Tipo (estandar/express): ").lower()
                peso = float(input("Peso (kg): "))
                dims = tuple(map(float, input("Dimensiones (largo,ancho,alto): ").split(",")))
                
                kwargs = {}
                if tipo == "express":
                    kwargs["urgencia"] = int(input("Urgencia (1-3): "))
                
                print("\nDESTINO:")
                calle = input("Calle: ")
                ciudad = input("Ciudad: ")
                cp = input("Código postal: ")
                pais = input("País: ")
                destino = Direccion(calle, ciudad, cp, pais)
                
                sistema.crear_envio(cliente, tipo, peso, dims, destino, **kwargs)
                print("✅ Envío creado")
                
            except Exception as e:
                print(f"❌ Error: {e}")
        
        elif opcion == "3":
            sistema.mostrar_envios()
        
        elif opcion == "4":
            sistema.generar_reporte()
        
        elif opcion == "5":
            if not sistema.envios:
                print("❌ No hay envíos registrados")
                continue
                
            sistema.mostrar_envios()
            try:
                envio_idx = int(input("Seleccione envío: ")) - 1
                envio = sistema.envios[envio_idx]
                nuevo_estado = input("Nuevo estado (pendiente/en tránsito/entregado): ")
                envio.paquete.set_estado(nuevo_estado)
                print("✅ Estado actualizado")
            except Exception as e:
                print(f"❌ Error: {e}")
        
        elif opcion == "6":
            sistema.guardar_datos()
            print("✅ Datos guardados")
        
        elif opcion == "7":
            print("¡Hasta pronto!")
            break
        
        else:
            print("❌ Opción inválida")

if __name__ == "__main__":
    menu_principal()